[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/05_binary-search.ipynb)

*DSA Interview Handbook · Notebook 5 of 14*

# Binary Search

Halve the search space every step — on sorted arrays and on the answer itself.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Guess the middle page of a dictionary and discard half. It also works on answers, not just on arrays.

**What interviewers really test here:** Off-by-ones, loop invariants, and binary search on the answer.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. The guessing game *(Basics)*
- [ ] 2. Two templates: exact match and first-true *(Core)*
- [ ] 3. Arrays that are only partly sorted *(Core)*
- [ ] 4. Binary search on the answer *(Advanced)*
- [ ] 5. Tricky cases *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

**Binary search** is a **divide-and-conquer** search algorithm that finds a target in a **sorted** sequence by repeatedly comparing it with the middle element and discarding the half that cannot contain it.



More generally, binary search finds the boundary of any **monotonic predicate** — a yes/no function over an ordered range whose answers look like `F F F … F T T … T`.

### Algorithm (iterative)

1. Set `low = 0`, `high = n − 1`.
2. While `low ≤ high`:

   1. `mid = low + (high − low) // 2`
   2. If `a[mid] == target` → return mid.
   3. If `a[mid] < target` → `low = mid + 1` (search the right half).
   4. Else → `high = mid − 1` (search the left half).
3. Return −1 (not found).



**Invariant**: if the target exists, it is always within `[low, high]`. Each iteration shrinks the range while keeping the invariant, so when the range is empty the target doesn't exist.



The `low + (high − low) // 2` form avoids integer overflow in fixed-width languages (Java, C++); in Python `(low + high) // 2` is fine.

### Complexity derivation

After k comparisons, the remaining range has about n / 2ᵏ elements. The search stops when that reaches 1:



```text
n / 2ᵏ = 1   →   2ᵏ = n   →   k = log₂ n
```



<table> <tr><th>Case</th><th>Time</th><th>Space</th></tr> <tr><td>Best (target at the first mid)</td><td>O(1)</td><td rowspan="3">O(1) iterative, O(log n) recursive (call stack)</td></tr> <tr><td>Average</td><td>O(log n)</td></tr> <tr><td>Worst</td><td>O(log n)</td></tr> </table>



Recurrence for the recursive version: `T(n) = T(n/2) + O(1)`, which solves to O(log n).

### Variants

| Variant | Finds | Python |
| --- | --- | --- |
| Exact search | any index of target | — |
| **Lower bound** | first index with a[i] ≥ x (insertion point) | `bisect.bisect_left(a, x)` |
| **Upper bound** | first index with a[i] &gt; x | `bisect.bisect_right(a, x)` |
| Count of x | upper − lower | `bisect_right − bisect_left` |
| Rotated array search | target in a rotated sorted array | check which half is sorted |
| Peak / bitonic | a local maximum | follow the rising slope |
| **Search on the answer** | smallest/largest feasible value | binary search a value range with a feasibility check |
| Real-valued (bisection) | roots, square roots | fixed number of iterations |

### Binary search vs linear search

|   | Linear search | Binary search |
| --- | --- | --- |
| Requirement | none | sorted data (or monotonic predicate) |
| Time | O(n) | O(log n) |
| Data structure | array or linked list | random-access structure (array); not a linked list |
| Best for | small or unsorted data, single search | large sorted data, many searches |



If you search only once in unsorted data, sorting first (O(n log n)) costs more than a linear scan. Binary search pays off with repeated queries.

### Advantages, disadvantages and applications

**👍 Pros**

##### Advantages

- Extremely fast: 30 steps for a billion elements
- O(1) extra space
- Generalises to any monotonic decision problem



**👎 Cons**

##### Disadvantages

- Needs sorted data and random access
- Keeping data sorted under inserts costs O(n) per insert in an array (use a balanced BST instead)
- Off-by-one errors are easy to make





#### Applications




- Searching in sorted databases and indexes; `git bisect` to find the commit that introduced a bug.
- Dictionary / phone book lookup; autocomplete over a sorted word list.
- Numerical methods (bisection method for equation roots).
- Capacity planning: minimum servers / bandwidth / speed that meets a deadline.

---
# Part 2: Lessons, step by step

## Lesson 1: The guessing game

*Level: Basics*

"I'm thinking of a number from 1 to 100." If you guess 1, 2, 3, … it can take 100 guesses. If you always guess the **middle** and I say "higher" or "lower", every guess throws away half of what's left:



```text
100 → 50 → 25 → 13 → 7 → 4 → 2 → 1      at most 7 guesses
```



That's **binary search**. The number of steps is how many times you can halve n: **log₂ n**.



| n | 1,000 | 1,000,000 | 1,000,000,000 |
| --- | --- | --- | --- |
| steps | 10 | 20 | 30 |



The only requirement: when you look at the middle, you must be able to say **which half can't contain the answer**. Sorted order gives you that.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def guessing_lesson():
    secret = 37
    lo, hi, guesses = 1, 100, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        guesses += 1
        if mid == secret:
            return guesses
        if mid < secret:
            lo = mid + 1
        else:
            hi = mid - 1


guessing_lesson()

3

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Guess #1: 50 is too high → keep the lower half
2. Guess #2: 25 is too low → keep the upper half
3. Guess #3: 37 — correct!

</details>


## Lesson 2: Two templates: exact match and first-true

*Level: Core*

#### 1. Exact match (closed range)



```python
lo, hi = 0, len(a) - 1
while lo <= hi:
    mid = (lo + hi) // 2
    if a[mid] == target: return mid
    if a[mid] < target: lo = mid + 1
    else:                hi = mid - 1
return -1
```



#### 2. First index where a condition is true (half-open)



Think of the array as answers to a yes/no question that look like `F F F T T T`. Find the first T.



```python
lo, hi = 0, len(a)          # answer is somewhere in [lo, hi]
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid    # mid could be the answer
    else:       lo = mid + 1
return lo
```



With `ok(i) = a[i] >= x` this is **lower bound** (first position ≥ x, Python's `bisect_left`). With `a[i] > x` it's **upper bound** (`bisect_right`). First and last position of x = these two.



**💡 Tip:** Template 2 is the one to master. Almost every binary search problem can be phrased as "find the first true".

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def bounds_lesson():
    a, x = [1, 3, 3, 3, 5, 8], 3
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] >= x:
            hi = mid
        else:
            lo = mid + 1
    return lo


bounds_lesson()

1

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Lower bound: is a[3] = 3 ≥ 3? True
2. Lower bound: is a[1] = 3 ≥ 3? True
3. Lower bound: is a[0] = 1 ≥ 3? False
4. First index with a[i] ≥ 3 is 1

</details>


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q5](#q5), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 3: Arrays that are only partly sorted

*Level: Core*

Binary search doesn't need a fully sorted array — it needs a way to discard half.




- **Rotated sorted array** (`[4,5,6,7,0,1,2]`): at any mid, at least one half is properly sorted. Check whether the target lies inside that sorted half's range; if yes go there, else go to the other half.
- **Minimum of a rotated array**: compare `a[mid]` with `a[hi]`. Bigger → the drop is to the right.
- **Peak element**: if `a[mid] < a[mid + 1]` you're walking uphill, so a peak exists to the right.
- **Duplicates** can hide which half is sorted; when `a[lo] == a[mid] == a[hi]`, shrink both ends by one (worst case becomes O(n)).

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q7](#q7), [Q11](#q11), [Q13](#q13), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 4: Binary search on the answer

*Level: Advanced*

The most powerful use of binary search isn't on an array at all — it's on the **answer**.



Example: "What's the slowest eating speed that finishes all bananas in h hours?" Speeds 1, 2, 3, … give answers that look like `no no no yes yes yes` — once a speed works, every faster speed works too. That's a monotonic yes/no question, so binary search the speed.




1. Find the smallest and largest possible answer (`lo`, `hi`).
2. Write `feasible(x)` — usually a simple greedy O(n) check.
3. Binary search for the first feasible x (or last, for "maximise").



Total cost: O(n · log(range)). Signals: "minimise the maximum", "maximise the minimum", "smallest capacity / speed / days such that…".

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def answer_space_lesson():
    piles, h = [3, 6, 7, 11], 8
    hours = lambda k: sum((p + k - 1) // k for p in piles)
    table = [hours(k) <= h for k in range(1, max(piles) + 1)]
    lo, hi = 1, max(piles)
    while lo < hi:
        k = (lo + hi) // 2
        if hours(k) <= h:
            hi = k
        else:
            lo = k + 1
    return lo


answer_space_lesson()

4

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Speed 6: needs 6 h → fits
2. Speed 3: needs 10 h → too slow
3. Speed 5: needs 8 h → fits
4. Speed 4: needs 8 h → fits
5. Slowest speed that fits: 4

</details>


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q10](#q10), [Q20](#q20), [Q14](#q14), [Q15](#q15), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 5: Tricky cases

*Level: Advanced*

- **Infinite loops**: if you ever write `lo = mid`, round the middle up: `mid = (lo + hi + 1) // 2`. Otherwise with `hi = lo + 1` you stay stuck.
- **Overflow**: in Java/C++ write `lo + (hi − lo) / 2`. Python ints don't overflow, but say it in interviews.
- **2-D matrix**: if rows continue each other, treat it as one sorted list (`k → (k // n, k % n)`). If only rows and columns are sorted, start at the top-right corner and step left or down — O(m + n).
- **Two sorted arrays (median)**: binary search the *cut* in the smaller array so that the left parts together hold half the elements and everything on the left ≤ everything on the right.
- **Square roots and real numbers**: search the value range; for decimals, loop a fixed number of times instead of `lo < hi`.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q8](#q8), [Q18](#q18), [Q19](#q19), [Q17](#q17). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Binary Search

🟢 Warm-up · *Closed range*

Return the index of `target` in sorted `nums`, or −1.

**Examples**

- `nums = [-1, 0, 3, 5, 9, 12], target = 9` → `4`
- `nums = [-1, 0, 3, 5, 9, 12], target = 2` → `-1`

In [ ]:
def search(nums, target):
    """
    Return the index of target in sorted nums, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search, [
    (([-1, 0, 3, 5, 9, 12], 9), 4),
    (([-1, 0, 3, 5, 9, 12], 2), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan left to right.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare with the middle; drop the half that can't contain target.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Scan left to right.

```python
def search(nums, target):
    for i, x in enumerate(nums):
        if x == target:
            return i
    return -1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Compare with the middle; drop the half that can't contain target.

```python
def search(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan left to right. That costs O(n) time and O(1) space."

"I can do better: Compare with the middle; drop the half that can't contain target. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [-1, 0, 3, 5, 9, 12, 15, 20, 31], target = 20`

1. nums[mid] = 9 vs 20
2. nums[mid] = 15 vs 20
3. nums[mid] = 20 vs 20

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/binary-search)

<a id="q2"></a>
### Q2. Search Insert Position

🟢 Warm-up · *Lower bound*

Return the index of `target` in sorted `nums`, or the index where it would be inserted.

**Examples**

- `nums = [1, 3, 5, 6], target = 5` → `2`
- `nums = [1, 3, 5, 6], target = 2` → `1`
- `nums = [1, 3, 5, 6], target = 7` → `4`

In [ ]:
def search_insert(nums, target):
    """
    Return the index of target in sorted nums, or the index where it would be
    inserted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search_insert, [
    (([1, 3, 5, 6], 5), 2),
    (([1, 3, 5, 6], 2), 1),
    (([1, 3, 5, 6], 7), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for the first element ≥ target.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find the first index with nums[i] ≥ target using the half-open template.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Scan for the first element ≥ target.

```python
def search_insert(nums, target):
    for i, x in enumerate(nums):
        if x >= target:
            return i
    return len(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Find the first index with nums[i] ≥ target using the half-open template.

```python
def search_insert(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for the first element ≥ target. That costs O(n) time and O(1) space."

"I can do better: Find the first index with nums[i] ≥ target using the half-open template. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 3, 5, 6, 8, 10, 12], target = 7`

1. nums[3] = 6 &lt; 7
2. nums[5] = 10 ≥ 7
3. nums[4] = 8 ≥ 7

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/search-insert-position)

<a id="q3"></a>
### Q3. Sqrt(x)

🟢 Warm-up · *Search on the answer*

Return ⌊√x⌋ without using a power or sqrt function.

**Examples**

- `x = 4` → `2`
- `x = 8` → `2`
- `x = 0` → `0`

In [ ]:
def my_sqrt(x):
    """
    Return ⌊√x⌋ without using a power or sqrt function.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(my_sqrt, [
    ((4,), 2),
    ((8,), 2),
    ((0,), 0),
    ((1,), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try r = 0, 1, 2, … while r² ≤ x.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find the largest r with r² ≤ x by binary search on [0, x].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try r = 0, 1, 2, … while r² ≤ x.

```python
def my_sqrt(x):
    r = 0
    while (r + 1) * (r + 1) <= x:
        r += 1
    return r
```

**Cost:** O(√x) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Find the largest r with r² ≤ x by binary search on [0, x].

```python
def my_sqrt(x):
    lo, hi = 0, x
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * mid <= x:
            lo = mid
        else:
            hi = mid - 1
    return lo
```

**Cost:** O(log x) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try r = 0, 1, 2, … while r² ≤ x. That costs O(√x) time and O(1) space."

"I can do better: Find the largest r with r² ≤ x by binary search on [0, x]. That's O(log x) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `x = 50`

1. 25² = 625 &gt; 50
2. 12² = 144 &gt; 50
3. 6² = 36 ≤ 50
4. 9² = 81 &gt; 50
5. 7² = 49 ≤ 50
6. 8² = 64 &gt; 50

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/sqrt-x)

<a id="q4"></a>
### Q4. Kth Missing Positive Number

🟢 Warm-up · *Missing count at index*

`arr` is strictly increasing positive integers. Return the k-th positive integer missing from it.

**Examples**

- `arr = [2, 3, 4, 7, 11], k = 5` → `9`
- `arr = [1, 2, 3, 4], k = 2` → `6`

In [ ]:
def find_kth_positive(arr, k):
    """
    arr is strictly increasing positive integers. Return the k-th positive
    integer missing from it.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_kth_positive, [
    (([2, 3, 4, 7, 11], 5), 9),
    (([1, 2, 3, 4], 2), 6),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count up from 1, skipping numbers in arr.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Before index i, arr[i] − (i + 1) numbers are missing. Binary search the first index where that count reaches k; the answer is index + k.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count up from 1, skipping numbers in arr.

```python
def find_kth_positive(arr, k):
    s, x = set(arr), 0
    while k:
        x += 1
        if x not in s:
            k -= 1
    return x
```

**Cost:** O(n + k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Before index i, arr[i] − (i + 1) numbers are missing. Binary search the first index where that count reaches k; the answer is index + k.

```python
def find_kth_positive(arr, k):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - mid - 1 >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo + k
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count up from 1, skipping numbers in arr. That costs O(n + k) time and O(1) space."

"I can do better: Before index i, arr[i] − (i + 1) numbers are missing. Binary search the first index where that count reaches k; the answer is index + k. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `arr = [2, 3, 4, 7, 11], k = 5`

1. Missing before index 2: 4 − 3 = 1
2. Missing before index 4: 11 − 5 = 6
3. Missing before index 3: 7 − 4 = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/kth-missing-positive-number)

<a id="q5"></a>
### Q5. Find First and Last Position of Element in Sorted Array

🟡 Interview standard · *Lower + upper bound*

Return `[first, last]` indices of `target` in sorted `nums`, or `[-1, -1]`.

**Examples**

- `nums = [5, 7, 7, 8, 8, 10], target = 8` → `[3, 4]`
- `nums = [5, 7, 7, 8, 8, 10], target = 6` → `[-1, -1]`
- `nums = [], target = 0` → `[-1, -1]`

In [ ]:
def search_range(nums, target):
    """
    Return [first, last] indices of target in sorted nums, or [-1, -1].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search_range, [
    (([5, 7, 7, 8, 8, 10], 8), [3, 4]),
    (([5, 7, 7, 8, 8, 10], 6), [-1, -1]),
    (([], 0), [-1, -1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for the first and last match.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two binary searches: first index ≥ target, and first index &gt; target (minus one).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Scan for the first and last match.

```python
def search_range(nums, target):
    idx = [i for i, x in enumerate(nums) if x == target]
    return [idx[0], idx[-1]] if idx else [-1, -1]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Two binary searches: first index ≥ target, and first index &gt; target (minus one).

```python
def search_range(nums, target):
    def first_at_least(x):
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] >= x:
                hi = mid
            else:
                lo = mid + 1
        return lo

    l = first_at_least(target)
    if l == len(nums) or nums[l] != target:
        return [-1, -1]
    r = first_at_least(target + 1) - 1
    return [l, r]
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for the first and last match. That costs O(n) time and O(1) space."

"I can do better: Two binary searches: first index ≥ target, and first index &gt; target (minus one). That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 4, 4, 4, 4, 7, 9], target = 4`

1. Looking for first index with value ≥ 4: nums[3] = 4
2. Looking for first index with value ≥ 4: nums[1] = 4
3. Looking for first index with value ≥ 4: nums[0] = 2
4. Looking for first index with value ≥ 5: nums[3] = 4
5. Looking for first index with value ≥ 5: nums[5] = 7
6. Looking for first index with value ≥ 5: nums[4] = 4
7. Range [1, 4]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/find-first-and-last-position-of-element-in-sorted-array)

<a id="q6"></a>
### Q6. Search in Rotated Sorted Array

🟡 Interview standard · *Find the sorted half*

A sorted array of distinct values was rotated at an unknown pivot. Return the index of `target` or −1 in O(log n).

**Examples**

- `nums = [4, 5, 6, 7, 0, 1, 2], target = 0` → `4`
- `nums = [4, 5, 6, 7, 0, 1, 2], target = 3` → `-1`
- `nums = [1], target = 0` → `-1`

In [ ]:
def search_rotated(nums, target):
    """
    A sorted array of distinct values was rotated at an unknown pivot. Return
    the index of target or −1 in O(log n).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search_rotated, [
    (([4, 5, 6, 7, 0, 1, 2], 0), 4),
    (([4, 5, 6, 7, 0, 1, 2], 3), -1),
    (([1], 0), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Linear scan.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

At any mid, one of the halves [lo, mid] or [mid, hi] is sorted. If target lies within the sorted half's range, search there; otherwise search the other half.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Linear scan.

```python
def search_rotated(nums, target):
    return nums.index(target) if target in nums else -1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** At any mid, one of the halves [lo, mid] or [mid, hi] is sorted. If target lies within the sorted half's range, search there; otherwise search the other half.

```python
def search_rotated(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Linear scan. That costs O(n) time and O(1) space."

"I can do better: At any mid, one of the halves [lo, mid] or [mid, hi] is sorted. If target lies within the sorted half's range, search there; otherwise search the other half. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [4, 5, 6, 7, 0, 1, 2], target = 0`

1. Left half 4..7 is sorted. Is 0 inside it?
2. Left half 0..1 is sorted. Is 0 inside it?
3. Found 0 at 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/search-in-rotated-sorted-array)

<a id="q7"></a>
### Q7. Find Minimum in Rotated Sorted Array

🟡 Interview standard · *Compare with right end*

Return the minimum of a rotated sorted array of distinct values in O(log n).

**Examples**

- `nums = [3, 4, 5, 1, 2]` → `1`
- `nums = [4, 5, 6, 7, 0, 1, 2]` → `0`
- `nums = [11, 13, 15, 17]` → `11`

In [ ]:
def find_min(nums):
    """
    Return the minimum of a rotated sorted array of distinct values in O(log n).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_min, [
    (([3, 4, 5, 1, 2],), 1),
    (([4, 5, 6, 7, 0, 1, 2],), 0),
    (([11, 13, 15, 17],), 11),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Take min() of the array.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

If nums[mid] &gt; nums[hi], the drop (the minimum) is to the right of mid; otherwise it is at mid or to its left.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Take min() of the array.

```python
def find_min(nums):
    return min(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** If nums[mid] &gt; nums[hi], the drop (the minimum) is to the right of mid; otherwise it is at mid or to its left.

```python
def find_min(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Take min() of the array. That costs O(n) time and O(1) space."

"I can do better: If nums[mid] &gt; nums[hi], the drop (the minimum) is to the right of mid; otherwise it is at mid or to its left. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [11, 13, 15, 17, 19, 2, 5, 7]`

1. nums[mid] = 17 &gt; nums[hi] = 7 → min is right of mid
2. nums[mid] = 2 ≤ nums[hi] = 7 → min is at or left of mid
3. nums[mid] = 19 &gt; nums[hi] = 2 → min is right of mid

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/find-minimum-in-rotated-sorted-array)

<a id="q8"></a>
### Q8. Search a 2D Matrix

🟡 Interview standard · *Flatten the index*

Each row is sorted and each row's first value is larger than the previous row's last. Is `target` in the matrix?

**Examples**

- `matrix = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], target = 3` → `True`
- `matrix = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], target = 13` → `False`

In [ ]:
def search_matrix(matrix, target):
    """
    Each row is sorted and each row's first value is larger than the previous
    row's last. Is target in the matrix?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search_matrix, [
    (([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 3), True),
    (([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 13), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every cell.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The matrix is one sorted list of m·n values. Binary search index k and read cell (k // n, k % n).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check every cell.

```python
def search_matrix(matrix, target):
    return any(target in row for row in matrix)
```

**Cost:** O(m · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The matrix is one sorted list of m·n values. Binary search index k and read cell (k // n, k % n).

```python
def search_matrix(matrix, target):
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        v = matrix[mid // n][mid % n]
        if v == target:
            return True
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False
```

**Cost:** O(log(m · n)) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every cell. That costs O(m · n) time and O(1) space."

"I can do better: The matrix is one sorted list of m·n values. Binary search index k and read cell (k // n, k % n). That's O(log(m · n)) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], target = 16`

1. Index 5 → cell (1, 1) = 11
2. Index 8 → cell (2, 0) = 23
3. Index 6 → cell (1, 2) = 16

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/search-a-2d-matrix)

<a id="q9"></a>
### Q9. Koko Eating Bananas

🟡 Interview standard · *Search on the answer*

Koko eats at speed k bananas per hour (one pile per hour at most). Return the minimum k that finishes all `piles` within `h` hours.

**Examples**

- `piles = [3, 6, 7, 11], h = 8` → `4`
- `piles = [30, 11, 23, 4, 20], h = 5` → `30`
- `piles = [30, 11, 23, 4, 20], h = 6` → `23`

In [ ]:
def min_eating_speed(piles, h):
    """
    Koko eats at speed k bananas per hour (one pile per hour at most). Return
    the minimum k that finishes all piles within h hours.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_eating_speed, [
    (([3, 6, 7, 11], 8), 4),
    (([30, 11, 23, 4, 20], 5), 30),
    (([30, 11, 23, 4, 20], 6), 23),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try k = 1, 2, 3, … until it fits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Hours needed only decrease as k grows, so binary search k in [1, max(piles)] for the first speed that finishes in time.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try k = 1, 2, 3, … until it fits.

```python
def min_eating_speed(piles, h):
    k = 1
    while sum((p + k - 1) // k for p in piles) > h:
        k += 1
    return k
```

**Cost:** O(max · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Hours needed only decrease as k grows, so binary search k in [1, max(piles)] for the first speed that finishes in time.

```python
def min_eating_speed(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        k = (lo + hi) // 2
        hours = sum((p + k - 1) // k for p in piles)
        if hours <= h:
            hi = k
        else:
            lo = k + 1
    return lo
```

**Cost:** O(n log max) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try k = 1, 2, 3, … until it fits. That costs O(max · n) time and O(1) space."

"I can do better: Hours needed only decrease as k grows, so binary search k in [1, max(piles)] for the first speed that finishes in time. That's O(n log max) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `piles = [3, 6, 7, 11], h = 8`

1. Speed 6: needs 6 h ≤ 8 → try slower
2. Speed 3: needs 10 h &gt; 8 → too slow
3. Speed 5: needs 8 h ≤ 8 → try slower
4. Speed 4: needs 8 h ≤ 8 → try slower

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/koko-eating-bananas)

<a id="q10"></a>
### Q10. Capacity To Ship Packages Within D Days

🟡 Interview standard · *Search on the answer*

Ship packages in order; each day the ship carries at most its capacity. Return the least capacity that ships everything within `days`.

**Examples**

- `weights = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], days = 5` → `15`
- `weights = [3, 2, 2, 4, 1, 4], days = 3` → `6`
- `weights = [1, 2, 3, 1, 1], days = 4` → `3`

In [ ]:
def ship_within_days(weights, days):
    """
    Ship packages in order; each day the ship carries at most its capacity.
    Return the least capacity that ships everything within days.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ship_within_days, [
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), 15),
    (([3, 2, 2, 4, 1, 4], 3), 6),
    (([1, 2, 3, 1, 1], 4), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every capacity from max(weights) upward.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Capacity lies in [max(weights), sum(weights)]. For a candidate, greedily fill days; binary search the smallest capacity that needs ≤ days.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every capacity from max(weights) upward.

```python
def ship_within_days(weights, days):
    def need(cap):
        d, load = 1, 0
        for w in weights:
            if load + w > cap:
                d, load = d + 1, 0
            load += w
        return d
    cap = max(weights)
    while need(cap) > days:
        cap += 1
    return cap
```

**Cost:** O(n · sum) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Capacity lies in [max(weights), sum(weights)]. For a candidate, greedily fill days; binary search the smallest capacity that needs ≤ days.

```python
def ship_within_days(weights, days):
    def need(cap):
        d, load = 1, 0
        for w in weights:
            if load + w > cap:
                d, load = d + 1, 0
            load += w
        return d

    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if need(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo
```

**Cost:** O(n log sum) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every capacity from max(weights) upward. That costs O(n · sum) time and O(1) space."

"I can do better: Capacity lies in [max(weights), sum(weights)]. For a candidate, greedily fill days; binary search the smallest capacity that needs ≤ days. That's O(n log sum) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `weights = [3, 2, 2, 4, 1, 4], days = 3`

1. Capacity 10 needs 2 day(s)
2. Capacity 7 needs 3 day(s)
3. Capacity 5 needs 4 day(s)
4. Capacity 6 needs 3 day(s)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/capacity-to-ship-packages-within-d-days)

<a id="q11"></a>
### Q11. Find Peak Element

🟡 Interview standard · *Follow the slope*

Return the index of any element strictly greater than its neighbours (treat outside as −∞), in O(log n).

**Examples**

- `nums = [1, 2, 3, 1]` → `2`
- `nums = [1, 2, 1, 3, 5, 6, 4]` → `5`

In [ ]:
def find_peak_element(nums):
    """
    Return the index of any element strictly greater than its neighbours (treat
    outside as −∞), in O(log n).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_peak_element, [(([1, 2, 3, 1],), 2), (([1, 2, 1, 3, 5, 6, 4],), 5)], verify=lambda a, i: 0 <= i < len(a[0]) and (i == 0 or a[0][i] > a[0][i - 1]) and (i == len(a[0]) - 1 or a[0][i] > a[0][i + 1]))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for the first i with nums[i] &gt; nums[i+1].

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

If nums[mid] &lt; nums[mid+1] we're on an uphill slope, so a peak exists to the right; otherwise one exists at mid or to the left.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Scan for the first i with nums[i] &gt; nums[i+1].

```python
def find_peak_element(nums):
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            return i
    return len(nums) - 1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** If nums[mid] &lt; nums[mid+1] we're on an uphill slope, so a peak exists to the right; otherwise one exists at mid or to the left.

```python
def find_peak_element(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for the first i with nums[i] &gt; nums[i+1]. That costs O(n) time and O(1) space."

"I can do better: If nums[mid] &lt; nums[mid+1] we're on an uphill slope, so a peak exists to the right; otherwise one exists at mid or to the left. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 1, 3, 5, 6, 4]`

1. 3 &lt; 5 → peak is right
2. 6 &gt; 4 → peak is here or left
3. 5 &lt; 6 → peak is right

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/find-peak-element)

<a id="q12"></a>
### Q12. Single Element in a Sorted Array

🟡 Interview standard · *Pair parity*

Every value appears twice except one. Find it in O(log n).

**Examples**

- `nums = [1, 1, 2, 3, 3, 4, 4, 8, 8]` → `2`
- `nums = [3, 3, 7, 7, 10, 11, 11]` → `10`

In [ ]:
def single_non_duplicate(nums):
    """
    Every value appears twice except one. Find it in O(log n).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(single_non_duplicate, [
    (([1, 1, 2, 3, 3, 4, 4, 8, 8],), 2),
    (([3, 3, 7, 7, 10, 11, 11],), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

XOR every element.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Before the single element, pairs start at even indices; after it, at odd ones. Check the pair at an even mid: if nums[mid] == nums[mid+1], the single is to the right.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** XOR every element.

```python
def single_non_duplicate(nums):
    x = 0
    for v in nums:
        x ^= v
    return x
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Before the single element, pairs start at even indices; after it, at odd ones. Check the pair at an even mid: if nums[mid] == nums[mid+1], the single is to the right.

```python
def single_non_duplicate(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if mid % 2:
            mid -= 1
        if nums[mid] == nums[mid + 1]:
            lo = mid + 2
        else:
            hi = mid
    return nums[lo]
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: XOR every element. That costs O(n) time and O(1) space."

"I can do better: Before the single element, pairs start at even indices; after it, at odd ones. Check the pair at an even mid: if nums[mid] == nums[mid+1], the single is to the right. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 1, 2, 3, 3, 4, 4, 8, 8]`

1. Pair at even index 4: 3, 4 → broken, go left
2. Pair at even index 2: 2, 3 → broken, go left
3. Pair at even index 0: 1, 1 → intact, go right

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/single-element-in-a-sorted-array)

<a id="q13"></a>
### Q13. Search in Rotated Sorted Array II

🟡 Interview standard · *Rotated with duplicates*

Like Search in Rotated Sorted Array, but values may repeat. Return whether `target` is present.

**Examples**

- `nums = [2, 5, 6, 0, 0, 1, 2], target = 0` → `True`
- `nums = [2, 5, 6, 0, 0, 1, 2], target = 3` → `False`
- `nums = [1, 0, 1, 1, 1], target = 0` → `True`

In [ ]:
def search_rotated_dup(nums, target):
    """
    Like Search in Rotated Sorted Array, but values may repeat. Return whether
    target is present.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search_rotated_dup, [
    (([2, 5, 6, 0, 0, 1, 2], 0), True),
    (([2, 5, 6, 0, 0, 1, 2], 3), False),
    (([1, 0, 1, 1, 1], 0), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Linear scan.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same sorted-half logic. When nums[lo] == nums[mid] == nums[hi] we can't tell which half is sorted, so shrink both ends by one.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Linear scan.

```python
def search_rotated_dup(nums, target):
    return target in nums
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Same sorted-half logic. When nums[lo] == nums[mid] == nums[hi] we can't tell which half is sorted, so shrink both ends by one.

```python
def search_rotated_dup(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo, hi = lo + 1, hi - 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False
```

**Cost:** O(log n) average, O(n) worst time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Linear scan. That costs O(n) time and O(1) space."

"I can do better: Same sorted-half logic. When nums[lo] == nums[mid] == nums[hi] we can't tell which half is sorted, so shrink both ends by one. That's O(log n) average, O(n) worst time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 0, 1, 1, 1], target = 0`

1. nums[mid] = 1
2. nums[mid] = 1
3. nums[mid] = 0

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/search-in-rotated-sorted-array-ii)

<a id="q14"></a>
### Q14. Minimum Number of Days to Make m Bouquets

🟡 Interview standard · *Search on the answer*

Flower i blooms on day `bloomDay[i]`. A bouquet needs `k` adjacent bloomed flowers. Return the earliest day you can make `m` bouquets, or −1.

**Examples**

- `bloom_day = [1, 10, 3, 10, 2], m = 3, k = 1` → `3`
- `bloom_day = [1, 10, 3, 10, 2], m = 3, k = 2` → `-1`
- `bloom_day = [7, 7, 7, 7, 12, 7, 7], m = 2, k = 3` → `12`

In [ ]:
def min_days(bloom_day, m, k):
    """
    Flower i blooms on day bloomDay[i]. A bouquet needs k adjacent bloomed
    flowers. Return the earliest day you can make m bouquets, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_days, [
    (([1, 10, 3, 10, 2], 3, 1), 3),
    (([1, 10, 3, 10, 2], 3, 2), -1),
    (([7, 7, 7, 7, 12, 7, 7], 2, 3), 12),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every distinct bloom day in increasing order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

More days never means fewer bouquets. Binary search the day; count bouquets by scanning runs of bloomed flowers.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every distinct bloom day in increasing order.

```python
def min_days(bloom_day, m, k):
    def count(day):
        b = run = 0
        for d in bloom_day:
            run = run + 1 if d <= day else 0
            if run == k:
                b, run = b + 1, 0
        return b
    for day in sorted(set(bloom_day)):
        if count(day) >= m:
            return day
    return -1
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** More days never means fewer bouquets. Binary search the day; count bouquets by scanning runs of bloomed flowers.

```python
def min_days(bloom_day, m, k):
    if m * k > len(bloom_day):
        return -1

    def count(day):
        b = run = 0
        for d in bloom_day:
            run = run + 1 if d <= day else 0
            if run == k:
                b, run = b + 1, 0
        return b

    lo, hi = min(bloom_day), max(bloom_day)
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= m:
            hi = mid
        else:
            lo = mid + 1
    return lo
```

**Cost:** O(n log max) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every distinct bloom day in increasing order. That costs O(n²) time and O(n) space."

"I can do better: More days never means fewer bouquets. Binary search the day; count bouquets by scanning runs of bloomed flowers. That's O(n log max) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `bloom_day = [1, 10, 3, 10, 2], m = 3, k = 1`

1. By day 5: 3 bouquet(s), need 3
2. By day 3: 3 bouquet(s), need 3
3. By day 2: 2 bouquet(s), need 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/minimum-number-of-days-to-make-m-bouquets)

<a id="q15"></a>
### Q15. Find the Smallest Divisor Given a Threshold

🟡 Interview standard · *Search on the answer*

Choose a positive divisor; sum ⌈nums[i] / divisor⌉. Return the smallest divisor with sum ≤ `threshold`.

**Examples**

- `nums = [1, 2, 5, 9], threshold = 6` → `5`
- `nums = [44, 22, 33, 11, 1], threshold = 5` → `44`

In [ ]:
def smallest_divisor(nums, threshold):
    """
    Choose a positive divisor; sum ⌈nums[i] / divisor⌉. Return the smallest
    divisor with sum ≤ threshold.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(smallest_divisor, [
    (([1, 2, 5, 9], 6), 5),
    (([44, 22, 33, 11, 1], 5), 44),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try divisors 1, 2, 3, … in order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The sum shrinks as the divisor grows. Binary search the divisor in [1, max(nums)].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try divisors 1, 2, 3, … in order.

```python
def smallest_divisor(nums, threshold):
    d = 1
    while sum((x + d - 1) // d for x in nums) > threshold:
        d += 1
    return d
```

**Cost:** O(n · max) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The sum shrinks as the divisor grows. Binary search the divisor in [1, max(nums)].

```python
def smallest_divisor(nums, threshold):
    lo, hi = 1, max(nums)
    while lo < hi:
        d = (lo + hi) // 2
        total = sum((x + d - 1) // d for x in nums)
        if total <= threshold:
            hi = d
        else:
            lo = d + 1
    return lo
```

**Cost:** O(n log max) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try divisors 1, 2, 3, … in order. That costs O(n · max) time and O(1) space."

"I can do better: The sum shrinks as the divisor grows. Binary search the divisor in [1, max(nums)]. That's O(n log max) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 5, 9], threshold = 6`

1. Divisor 5: sum of ceilings 5 vs threshold 6
2. Divisor 3: sum of ceilings 7 vs threshold 6
3. Divisor 4: sum of ceilings 7 vs threshold 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/find-the-smallest-divisor-given-a-threshold)

<a id="q16"></a>
### Q16. Magnetic Force Between Two Balls

🟡 Interview standard · *Maximise the min (aggressive cows)*

Place `m` balls in baskets at `position` so the minimum distance between any two balls is as large as possible. Return that distance.

**Examples**

- `position = [1, 2, 3, 4, 7], m = 3` → `3`
- `position = [5, 4, 3, 2, 1, 1000000000], m = 2` → `999999999`

In [ ]:
def max_distance(position, m):
    """
    Place m balls in baskets at position so the minimum distance between any two
    balls is as large as possible. Return that distance.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_distance, [
    (([1, 2, 3, 4, 7], 3), 3),
    (([5, 4, 3, 2, 1, 1000000000], 2), 999999999),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every gap d from 1 upward and check greedily.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort. For a gap d, place balls greedily left to right. If m balls fit, a larger d may too, so binary search the largest feasible d.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every gap d from 1 upward and check greedily.

```python
def max_distance(position, m):
    p = sorted(position)

    def fits(d):
        count, last = 1, p[0]
        for x in p[1:]:
            if x - last >= d:
                count, last = count + 1, x
        return count >= m

    best = 0
    for d in sorted({b - a for i, a in enumerate(p) for b in p[i + 1:]}):
        if fits(d):
            best = d
    return best
```

**Cost:** O(n · range) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort. For a gap d, place balls greedily left to right. If m balls fit, a larger d may too, so binary search the largest feasible d.

```python
def max_distance(position, m):
    position.sort()

    def fits(d):
        count, last = 1, position[0]
        for x in position[1:]:
            if x - last >= d:
                count, last = count + 1, x
        return count >= m

    lo, hi = 1, position[-1] - position[0]
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if fits(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo
```

**Cost:** O(n log range) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every gap d from 1 upward and check greedily. That costs O(n · range) time and O(1) space."

"I can do better: Sort. For a gap d, place balls greedily left to right. If m balls fit, a larger d may too, so binary search the largest feasible d. That's O(n log range) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `position = [1, 2, 3, 4, 7], m = 3`

1. Can 3 balls be at least 4 apart? False
2. Can 3 balls be at least 2 apart? True
3. Can 3 balls be at least 3 apart? True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/magnetic-force-between-two-balls)

<a id="q17"></a>
### Q17. Find K Closest Elements

🟡 Interview standard · *Binary search the window start*

Return the `k` elements of sorted `arr` closest to `x`, in ascending order (ties prefer the smaller value).

**Examples**

- `arr = [1, 2, 3, 4, 5], k = 4, x = 3` → `[1, 2, 3, 4]`
- `arr = [1, 1, 2, 3, 4, 5], k = 4, x = -1` → `[1, 1, 2, 3]`

In [ ]:
def find_closest_elements(arr, k, x):
    """
    Return the k elements of sorted arr closest to x, in ascending order (ties
    prefer the smaller value).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_closest_elements, [
    (([1, 2, 3, 4, 5], 4, 3), [1, 2, 3, 4]),
    (([1, 1, 2, 3, 4, 5], 4, -1), [1, 1, 2, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort by (|a − x|, a), take k, sort again.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Binary search the left edge L of the k-window in [0, n − k]: if x − arr[mid] &gt; arr[mid + k] − x, the window should move right.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort by (|a − x|, a), take k, sort again.

```python
def find_closest_elements(arr, k, x):
    return sorted(sorted(arr, key=lambda a: (abs(a - x), a))[:k])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Binary search the left edge L of the k-window in [0, n − k]: if x − arr[mid] &gt; arr[mid + k] − x, the window should move right.

```python
def find_closest_elements(arr, k, x):
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        if x - arr[mid] > arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo:lo + k]
```

**Cost:** O(log(n − k) + k) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort by (|a − x|, a), take k, sort again. That costs O(n log n) time and O(n) space."

"I can do better: Binary search the left edge L of the k-window in [0, n − k]: if x − arr[mid] &gt; arr[mid + k] − x, the window should move right. That's O(log(n − k) + k) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `arr = [1, 3, 5, 8, 9, 12, 15, 18, 20], k = 3, x = 10`

1. Window starting at 3: compare x − 8 = 2 with 15 − x = 5
2. Window starting at 1: compare x − 3 = 7 with 9 − x = -1
3. Window starting at 2: compare x − 5 = 5 with 12 − x = 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/find-k-closest-elements)

<a id="q18"></a>
### Q18. Search a 2D Matrix II

🟡 Interview standard · *Staircase from a corner*

Rows and columns are each sorted ascending. Return whether `target` is in the matrix.

**Examples**

- `matrix = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]], target = 5` → `True`
- `matrix = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]], target = 20` → `False`

In [ ]:
def search_matrix_ii(matrix, target):
    """
    Rows and columns are each sorted ascending. Return whether target is in the
    matrix.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(search_matrix_ii, [
    (([[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]], 5), True),
    (([[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]], 20), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Binary search each row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Start at the top-right corner. Too big → move left (the whole column below is bigger). Too small → move down. Each step removes a row or a column.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Binary search each row.

```python
def search_matrix_ii(matrix, target):
    from bisect import bisect_left
    for row in matrix:
        i = bisect_left(row, target)
        if i < len(row) and row[i] == target:
            return True
    return False
```

**Cost:** O(m log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Start at the top-right corner. Too big → move left (the whole column below is bigger). Too small → move down. Each step removes a row or a column.

```python
def search_matrix_ii(matrix, target):
    r, c = 0, len(matrix[0]) - 1
    path = []
    while r < len(matrix) and c >= 0:
        v = matrix[r][c]
        path.append((r, c))
        if v == target:
            return True
        if v > target:
            c -= 1
        else:
            r += 1
    return False
```

**Cost:** O(m + n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Binary search each row. That costs O(m log n) time and O(1) space."

"I can do better: Start at the top-right corner. Too big → move left (the whole column below is bigger). Too small → move down. Each step removes a row or a column. That's O(m + n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]], target = 14`

1. (0,4) = 15 &gt; target → left
2. (0,3) = 11 &lt; target → down
3. (1,3) = 12 &lt; target → down
4. (2,3) = 16 &gt; target → left
5. (2,2) = 9 &lt; target → down
6. (3,2) = 14 = target

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/search-a-2d-matrix-ii)

<a id="q19"></a>
### Q19. Median of Two Sorted Arrays

🔴 Stretch · *Binary search the partition*

Return the median of two sorted arrays in O(log(min(m, n))).

**Examples**

- `a = [1, 3], b = [2]` → `2.0`
- `a = [1, 2], b = [3, 4]` → `2.5`
- `a = [0, 0], b = [0, 0]` → `0.0`

In [ ]:
def find_median_sorted_arrays(a, b):
    """
    Return the median of two sorted arrays in O(log(min(m, n))).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_median_sorted_arrays, [
    (([1, 3], [2]), 2.0),
    (([1, 2], [3, 4]), 2.5),
    (([0, 0], [0, 0]), 0.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Merge both arrays and take the middle.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Cut A at i and B at j so the left side has half of all elements. The cut is right when A[i−1] ≤ B[j] and B[j−1] ≤ A[i]; binary search i over the shorter array.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Merge both arrays and take the middle.

```python
def find_median_sorted_arrays(a, b):
    m = sorted(a + b)
    n = len(m)
    return float(m[n // 2]) if n % 2 else (m[n // 2 - 1] + m[n // 2]) / 2
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Cut A at i and B at j so the left side has half of all elements. The cut is right when A[i−1] ≤ B[j] and B[j−1] ≤ A[i]; binary search i over the shorter array.

```python
def find_median_sorted_arrays(a, b):
    if len(a) > len(b):
        a, b = b, a
    m, n = len(a), len(b)
    half = (m + n + 1) // 2
    lo, hi = 0, m
    while lo <= hi:
        i = (lo + hi) // 2
        j = half - i
        aL = a[i - 1] if i > 0 else float("-inf")
        aR = a[i] if i < m else float("inf")
        bL = b[j - 1] if j > 0 else float("-inf")
        bR = b[j] if j < n else float("inf")
        if aL <= bR and bL <= aR:
            if (m + n) % 2:
                return float(max(aL, bL))
            return (max(aL, bL) + min(aR, bR)) / 2
        if aL > bR:
            hi = i - 1
        else:
            lo = i + 1
```

**Cost:** O(log min(m, n)) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Merge both arrays and take the middle. That costs O(m + n) time and O(m + n) space."

"I can do better: Cut A at i and B at j so the left side has half of all elements. The cut is right when A[i−1] ≤ B[j] and B[j−1] ≤ A[i]; binary search i over the shorter array. That's O(log min(m, n)) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/median-of-two-sorted-arrays)

<a id="q20"></a>
### Q20. Split Array Largest Sum

🔴 Stretch · *Minimise the max*

Split `nums` into `k` non-empty contiguous parts to minimise the largest part sum.

**Examples**

- `nums = [7, 2, 5, 10, 8], k = 2` → `18`
- `nums = [1, 2, 3, 4, 5], k = 2` → `9`
- `nums = [1, 4, 4], k = 3` → `4`

In [ ]:
def split_array(nums, k):
    """
    Split nums into k non-empty contiguous parts to minimise the largest part
    sum.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_array, [
    (([7, 2, 5, 10, 8], 2), 18),
    (([1, 2, 3, 4, 5], 2), 9),
    (([1, 4, 4], 3), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every placement of k − 1 cuts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Guess a cap S. Greedily pack elements until adding one would exceed S; count the parts. Fewer parts are needed for bigger S, so binary search the smallest S that needs ≤ k parts.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every placement of k − 1 cuts.

```python
def split_array(nums, k):
    from itertools import combinations
    n, best = len(nums), float("inf")
    for cuts in combinations(range(1, n), k - 1):
        b = (0,) + cuts + (n,)
        best = min(best, max(sum(nums[b[i]:b[i + 1]]) for i in range(k)))
    return best
```

**Cost:** O(C(n, k) · n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Guess a cap S. Greedily pack elements until adding one would exceed S; count the parts. Fewer parts are needed for bigger S, so binary search the smallest S that needs ≤ k parts.

```python
def split_array(nums, k):
    def parts(cap):
        count, cur = 1, 0
        for x in nums:
            if cur + x > cap:
                count, cur = count + 1, 0
            cur += x
        return count

    lo, hi = max(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if parts(mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo
```

**Cost:** O(n log sum) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every placement of k − 1 cuts. That costs O(C(n, k) · n) time and O(k) space."

"I can do better: Guess a cap S. Greedily pack elements until adding one would exceed S; count the parts. Fewer parts are needed for bigger S, so binary search the smallest S that needs ≤ k parts. That's O(n log sum) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [7, 2, 5, 10, 8], k = 2`

1. Max part sum 21 → needs 2 part(s), allowed 2
2. Max part sum 15 → needs 3 part(s), allowed 2
3. Max part sum 18 → needs 2 part(s), allowed 2
4. Max part sum 17 → needs 3 part(s), allowed 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/binary-search/split-array-largest-sum)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why does binary search need a sorted array?</b></summary>

Discarding half depends on knowing every element on one side of mid is smaller (or larger) than the target. Without sorted order, comparing with mid gives no information about the rest.

</details>

<details>
<summary><b>2. Can binary search be used on a linked list?</b></summary>

Technically yes, but reaching the middle node takes O(n), so total time is O(n) — no better than linear search. Binary search needs O(1) random access.

</details>

<details>
<summary><b>3. What is the time complexity and why?</b></summary>

O(log n): each comparison halves the remaining range, so after k steps n / 2ᵏ elements remain; k = log₂ n steps reduce it to one.

</details>

<details>
<summary><b>4. What is 'binary search on the answer'?</b></summary>

When the answer lies in a range and feasibility is monotonic (if x works, every larger x works), binary search over candidate answers, checking each with a feasibility function. Cost: O(log(range) × cost of the check).

</details>

<details>
<summary><b>5. How do you avoid infinite loops?</b></summary>

Make sure the range shrinks every iteration. With `lo = mid` use the upper middle `(lo + hi + 1) // 2`; with `hi = mid` use the lower middle. Pick one template (closed or half-open) and stay consistent.

</details>

<details>
<summary><b>6. Recursive or iterative binary search?</b></summary>

Both are O(log n) time. Iterative uses O(1) space; recursive uses O(log n) stack space. Iterative is preferred in practice.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: The guessing game
- [ ] I can explain: Two templates: exact match and first-true
- [ ] I can explain: Arrays that are only partly sorted
- [ ] I can explain: Binary search on the answer
- [ ] I can explain: Tricky cases

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Magnetic Force Between Two Balls
- [ ] [Q17](#q17) Find K Closest Elements
- [ ] [Q18](#q18) Search a 2D Matrix II
- [ ] [Q19](#q19) Median of Two Sorted Arrays
- [ ] [Q20](#q20) Split Array Largest Sum

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.